In [5]:
import logging
import time
from typing import Any, Dict, Union, List

from langgraph.types import Command
from langgraph.graph import END, START

from src.haive.core.engine.agent.agent import Agent, AgentConfig, register_agent
from src.haive.agents.rag.base.config import BaseRAGConfig, BaseRAGState
from src.haive.core.models.retriever.base import RetrieverConfig
from src.haive.core.models.vectorstore.base import VectorStoreConfig
from src.haive.core.engine.aug_llm import AugLLMConfig
from src.haive.core.graph.GraphBuilder import DynamicGraph
from src.haive.core.graph.branches import Branch

logger = logging.getLogger(__name__)

@register_agent(BaseRAGConfig)
class BaseRAGAgent(Agent[BaseRAGConfig]):
    """
    Simplified RAG agent that focuses on document retrieval only.
    
    This agent implements a basic workflow:
    1. Receive a query
    2. Retrieve relevant documents
    """

    def __init__(self, config: BaseRAGConfig):
        super().__init__(config)
        # Initialize retriever
        self._retriever = None

    @property
    def retriever(self):
        """Lazy initialization of retriever."""
        if self._retriever is None:
            self._retriever = self.config.retriever_config.create_runnable()
        return self._retriever

    def setup_workflow(self) -> None:
        """Set up the retrieval workflow graph."""
        # Just one node for retrieval in this simplified version
        self.graph.add_node("retrieve", self.retrieve_documents)
        
        # Simple workflow: START -> retrieve -> END
        self.graph.add_edge(START, "retrieve")
        self.graph.add_edge("retrieve", END)
        
        logger.info(f"Basic retrieval workflow set up for {self.config.name}")

    def retrieve_documents(self, state: BaseRAGState) -> Command:
        """
        Retrieve relevant documents based on the query.
        
        Args:
            state: Current state with query
            
        Returns:
            Command for updating state with retrieved documents
        """
        logger.info(f"Retrieving documents for query: {state.query}")
        start_time = time.time()
        
        try:
            # Use retriever to get documents
            documents = self.retriever.invoke(state.query)
            print('----------DOCUMENTS-----------------')
            print(documents)
            # Convert documents to dictionary format if needed
            #doc_list = []
            
            
            logger.info(f"Retrieved {len(documents)} documents in {time.time() - start_time:.2f}s")
            
            # Update state with retrieved documents
            return Command(
                update={"retrieved_documents": documents}
            )
        
        except Exception as e:
            logger.error(f"Error retrieving documents: {str(e)}")
            return Command(
                update={"error": f"Error retrieving documents: {str(e)}"}
            )

In [8]:
"""
Example of using the RAG agent with the new architecture.
"""

import logging
from langchain_community.document_loaders import WebBaseLoader

# Import from our architecture
from src.haive.core.models.embeddings.base import HuggingFaceEmbeddingConfig
from src.haive.core.engine.vectorstore import VectorStoreConfig
from src.haive.core.engine.retriever import VectorStoreRetrieverConfig
from src.haive.agents.rag.base.config import BaseRAGConfig


import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Clear any existing handlers
if logger.hasHandlers():
    logger.handlers.clear()

# Add a basic stream handler with formatting
handler = logging.StreamHandler()
formatter = logging.Formatter('[%(levelname)s] %(name)s: %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

#def main():
# 1. Load documents
logger.info("Loading documents...")
docs = WebBaseLoader("https://langchain.com/docs/").load()
logger.info(f"Loaded {len(docs)} documents")

# 2. Create VectorStore config
logger.info("Creating vector store...")
vs_config = VectorStoreConfig(
    name="langchain_docs_vectorstore",
    documents=docs,
    embedding_model=HuggingFaceEmbeddingConfig(model="sentence-transformers/all-mpnet-base-v2")
)

# 3. Create retriever config (Method 1: explicitly)
logger.info("Creating retriever config (Method 1)...")
retriever_config = VectorStoreRetrieverConfig(
    name="langchain_docs_retriever",
    vector_store_config=vs_config,
    k=3
)

# Method 2: Pass vector store directly to RAG config
logger.info("Creating RAG agent (Method 2)...")
rag_agent_1 = BaseRAGAgent(
    BaseRAGConfig(
        name="direct_vectorstore_rag",
        retriever_config=vs_config
    )
)

logger.info("Creating RAG agent with explicit retriever...")
rag_agent_2 = BaseRAGAgent(
    BaseRAGConfig(
        name="explicit_retriever_rag",
        retriever_config=retriever_config
    )
)

# 4. Run queries
query = "What is LangChain?"

logger.info(f"Running query with agent 1: {query}")
result_1 = rag_agent_1.run({"query": query})
state_1 = result_1.values  # ✅ FIXED

logger.info(f"Running query with agent 2: {query}")
result_2 = rag_agent_2.run({"query": query})
state_2 = result_2.values  # ✅ FIXED

# Log outputs
logger.info(f"Retrieved {len(state_1['retrieved_documents'])} documents (Agent 1)")
logger.info(f"Retrieved {len(state_2['retrieved_documents'])} documents (Agent 2)")

if state_1['retrieved_documents']:
    logger.info("First document from agent 1:")
    logger.info(state_1['retrieved_documents'][0].page_content[:200] + "...")

if state_2['retrieved_documents']:
    logger.info("First document from agent 2:")
    logger.info(state_2['retrieved_documents'][0].page_content[:200] + "...")

    return state_1, state_2


#if __name__ == "__main__":
    #main()


[INFO] root: Loading documents...
[INFO] root: Loaded 1 documents
[INFO] root: Creating vector store...
[INFO] root: Creating retriever config (Method 1)...
[INFO] root: Creating RAG agent (Method 2)...
[INFO] src.haive.core.engine.agent.agent: No persistence config provided for direct_vectorstore_rag. Using memory checkpointer.
[INFO] root: Basic retrieval workflow set up for direct_vectorstore_rag
[INFO] src.haive.core.engine.agent.agent: Workflow compiled successfully for direct_vectorstore_rag
[INFO] src.haive.core.engine.agent.agent: Graph visualization saved to /home/will/Projects/haive/backend/haive/resources/Graphs/direct_vectorstore_rag_20250404_135942.png
[INFO] root: Creating RAG agent with explicit retriever...
[INFO] src.haive.core.engine.agent.agent: No persistence config provided for explicit_retriever_rag. Using memory checkpointer.
[INFO] root: Basic retrieval workflow set up for explicit_retriever_rag
[INFO] src.haive.core.engine.agent.agent: Workflow compiled success

Debug - Attempting to instantiate Azure OpenAI model:
- Model/deployment: gpt-4o
- API version: 2024-08-01-preview
- API base: https://awt-gpt.openai.azure.com/
- API type: azure
- API key available: Yes
Graph diagram saved as /home/will/Projects/haive/backend/haive/resources/Graphs/direct_vectorstore_rag_20250404_135942.png
Debug - Attempting to instantiate Azure OpenAI model:
- Model/deployment: gpt-4o
- API version: 2024-08-01-preview
- API base: https://awt-gpt.openai.azure.com/
- API type: azure
- API key available: Yes


[INFO] src.haive.core.engine.agent.agent: Graph visualization saved to /home/will/Projects/haive/backend/haive/resources/Graphs/explicit_retriever_rag_20250404_135942.png
[INFO] root: Running query with agent 1: What is LangChain?
[INFO] src.haive.core.engine.agent.agent: Running agent direct_vectorstore_rag with input: query='What is LangChain?' retrieved_documents=[] answer=None error=None metadata={}
[INFO] root: Retrieving documents for query: What is LangChain?
[INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2


Graph diagram saved as /home/will/Projects/haive/backend/haive/resources/Graphs/explicit_retriever_rag_20250404_135942.png


[INFO] src.haive.core.engine.retriever: Created VectorStoreRetriever 'retriever_for_direct_vectorstore_rag' with search_type=similarity
[INFO] root: Retrieved 1 documents in 1.56s
[INFO] src.haive.core.engine.agent.agent: State history saved to: /home/will/Projects/haive/backend/haive/resources/State_History/direct_vectorstore_rag_20250404_135942.json
[INFO] root: Running query with agent 2: What is LangChain?
[INFO] src.haive.core.engine.agent.agent: Running agent explicit_retriever_rag with input: query='What is LangChain?' retrieved_documents=[] answer=None error=None metadata={}
[INFO] root: Retrieving documents for query: What is LangChain?
[INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2


----------DOCUMENTS-----------------
[Document(id='47eef065-5606-4f5c-9ebd-e5d316505aea', metadata={'source': 'https://langchain.com/docs/', 'title': 'Introduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='\n\n\n\n\nIntroduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain\n\n\n\n\n\n\nSkip to main contentJoin us at  Interrupt: The Agent AI Conference by LangChain on May 13 & 14 in San Francisco!IntegrationsAPI ReferenceMoreContributingPeopleError referenceLangSmithLangGraphLangChain HubLangChain JS/TSv0.3v0.3v0.2v0.1\uf8ffüí¨SearchIntroductionTutorialsBuild a Question Answering application over a Graph DatabaseTutorialsBuild a simple LLM application with chat models and prompt templatesBuild a ChatbotBuild a Retrieval Augmented Generation (RAG) App: Part 2Build an Extraction ChainBuild an AgentTaggingBuild a Retrieval Augmented Generation (RAG) App: Part 1Bui

[INFO] src.haive.core.engine.retriever: Created VectorStoreRetriever 'langchain_docs_retriever' with search_type=similarity
[INFO] root: Retrieved 1 documents in 1.27s
[INFO] src.haive.core.engine.agent.agent: State history saved to: /home/will/Projects/haive/backend/haive/resources/State_History/explicit_retriever_rag_20250404_135942.json
[INFO] root: Retrieved 1 documents (Agent 1)
[INFO] root: Retrieved 1 documents (Agent 2)
[INFO] root: First document from agent 1:
[INFO] root: 




Introduction | ü¶úÔ∏èüîó LangChain






Skip to main contentJoin us at  Interrupt: The Agent AI Conference by LangChain on May 13 & 14 in San Francisco!IntegrationsAPI ReferenceMoreContributin...


----------DOCUMENTS-----------------
[Document(id='20e38696-1b62-46e8-9428-125f99ea3bf1', metadata={'source': 'https://langchain.com/docs/', 'title': 'Introduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='\n\n\n\n\nIntroduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain\n\n\n\n\n\n\nSkip to main contentJoin us at  Interrupt: The Agent AI Conference by LangChain on May 13 & 14 in San Francisco!IntegrationsAPI ReferenceMoreContributingPeopleError referenceLangSmithLangGraphLangChain HubLangChain JS/TSv0.3v0.3v0.2v0.1\uf8ffüí¨SearchIntroductionTutorialsBuild a Question Answering application over a Graph DatabaseTutorialsBuild a simple LLM application with chat models and prompt templatesBuild a ChatbotBuild a Retrieval Augmented Generation (RAG) App: Part 2Build an Extraction ChainBuild an AgentTaggingBuild a Retrieval Augmented Generation (RAG) App: Part 1Bui

SyntaxError: 'return' outside function (1750990026.py, line 92)